# Stage-by-Stage Model Evaluation

Loads every `*_stage.pt` checkpoint saved during training and runs:
1. **Architecture check** — forward pass, finite logits, causal mask
2. **Perplexity / BPC** — WikiText-2 test set (lower is better)
3. **Top-k accuracy** — next-token prediction on WikiText-2 val
4. **Generation samples** — stage-appropriate prompts
5. **BLiMP / LAMBADA** — grammaticality & long-range context (optional, slower)

Run all cells top-to-bottom after training completes.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Must match the values used during training (Cell 3 of kaggle_dual_gpu_finetune.ipynb)

import sys
from pathlib import Path

# Where training wrote its outputs
OUTPUT_DIR = "/kaggle/working/slm_run"

# Model architecture — must match training config exactly
MODEL_CFG = dict(
    dim     = 512,
    depth   = 8,
    heads   = 8,
    mlp_dim = 2048,
    window  = 2048,
    kv_heads = 2,
    dropout  = 0.0,   # always 0 at eval time
)

# Tokenizer — set TOKENIZER_PATH to a HybridTokenizer .pkl.gz, or leave None for GPT-2 BPE
TOKENIZER_PATH = None
HF_TOKENIZER   = "gpt2"

# REPO_ROOT: auto-detected (same logic as training notebook)
_repo_candidate = "/kaggle/input/small-language-model"
REPO_ROOT = _repo_candidate if Path(_repo_candidate).is_dir() else None
if REPO_ROOT:
    for _sub in ("src", "tests"):
        _p = str(Path(REPO_ROOT) / _sub)
        if _p not in sys.path:
            sys.path.insert(0, _p)
    print(f"[REPO_ROOT] {REPO_ROOT}")
else:
    print("[REPO_ROOT] Not found — using pip-installed package.")

# How many eval examples to use (reduce for speed, increase for accuracy)
QUICK = True   # True = 20% samples (~2-3 min/stage); False = full run (~10 min/stage)

# Generation prompts used for each stage (matched by substring of stage name)
STAGE_PROMPTS = {
    "tinystories": "Once upon a time, there was a little",
    "stories":     "Once upon a time, there was a little",
    "wikitext":    "The history of the Roman Empire began",
    "openwebtext": "Scientists have recently discovered that",
    "c4":          "The best way to learn a new skill is",
    "dolly":       "### Instruction:\nExplain what gravity is in simple terms.\n\n### Response:",
    "alpaca":      "### Instruction:\nExplain what gravity is in simple terms.\n\n### Response:",
    "gsm8k":       "John has 5 apples. He gives 2 to Mary. How many apples does John have left?",
    "openorca":    "Question: What is the capital of France? Answer:",
    "default":     "The meaning of life is",
}
GEN_MAX_TOKENS = 80
GEN_TEMPERATURE = 0.7

In [ ]:
# ── Discover checkpoints ──────────────────────────────────────────────────────
import os

out = Path(OUTPUT_DIR)
if not out.is_dir():
    raise FileNotFoundError(f"OUTPUT_DIR not found: {out}\nRun training first.")

# Stage checkpoints (*_stage.pt) — one per completed training stage
stage_ckpts = sorted(out.glob("*_stage.pt"), key=lambda p: p.stat().st_mtime)

# Latest step checkpoint (checkpoint-N/state.pt) — full model+optimizer state
step_ckpts = sorted(
    [d / "state.pt" for d in out.iterdir()
     if d.is_dir() and d.name.startswith("checkpoint-") and (d / "state.pt").exists()],
    key=lambda p: int(p.parent.name.split("-")[1]),
)

# Final model weights (if training finished all stages)
final_ckpt = out / "final_model.pt"

print(f"OUTPUT_DIR: {out}")
print(f"\nStage checkpoints ({len(stage_ckpts)} found):")
for p in stage_ckpts:
    sz = p.stat().st_size / 1024**2
    print(f"  {p.name:<40}  {sz:.0f} MB")

print(f"\nStep checkpoints ({len(step_ckpts)} found):")
for p in step_ckpts:
    sz = p.stat().st_size / 1024**2
    print(f"  {p.parent.name}/{p.name:<30}  {sz:.0f} MB")

if final_ckpt.exists():
    print(f"\nFinal model: {final_ckpt}  ({final_ckpt.stat().st_size/1024**2:.0f} MB)")
else:
    print("\nFinal model: not yet saved")

if not stage_ckpts and not step_ckpts:
    print("\nNo checkpoints found. Run at least one training stage first.")

In [ ]:
# ── Model + tokenizer loader ──────────────────────────────────────────────────
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load tokenizer once (shared across all stages)
if TOKENIZER_PATH and Path(TOKENIZER_PATH).exists():
    from my_slm.hybrid_tokeniztion import HybridTokenizer
    tokenizer = HybridTokenizer.load(TOKENIZER_PATH)
    vocab_size = tokenizer.vocab_size
    print(f"Tokenizer: HybridTokenizer  vocab={vocab_size:,}")
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.model_max_length = int(1e9)   # suppress length warnings
    vocab_size = len(tokenizer)
    print(f"Tokenizer: {HF_TOKENIZER}  vocab={vocab_size:,}")


def load_stage_model(ckpt_path: Path) -> torch.nn.Module:
    """Load a *_stage.pt or checkpoint-N/state.pt into a Transformer."""
    from my_slm.transformer import Transformer

    ckpt = torch.load(ckpt_path, map_location="cpu")
    state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt

    # Stage checkpoints have empty config — use notebook MODEL_CFG
    saved_cfg = ckpt.get("config", {})
    cfg = {**MODEL_CFG, **{k: v for k, v in saved_cfg.items() if v}}  # saved takes priority if non-empty

    model = Transformer(
        vocab_size = cfg.get("vocab_size", vocab_size),
        **{k: cfg[k] for k in ("dim", "depth", "heads", "mlp_dim", "window", "kv_heads", "dropout")},
    ).to(device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  [warn] missing keys: {missing[:5]}")
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    print(f"  Loaded {ckpt_path.name}  ({params/1e6:.1f}M params)")
    return model


def pick_prompt(stage_name: str) -> str:
    """Choose a generation prompt appropriate for the stage's training data."""
    stage_lower = stage_name.lower()
    for key, prompt in STAGE_PROMPTS.items():
        if key in stage_lower:
            return prompt
    return STAGE_PROMPTS["default"]


@torch.no_grad()
def generate(model, prompt: str, max_new_tokens: int = GEN_MAX_TOKENS,
             temperature: float = GEN_TEMPERATURE) -> str:
    """Greedy/sampled generation from a text prompt."""
    if hasattr(tokenizer, 'encode') and hasattr(tokenizer, 'token2id'):
        ids = tokenizer.encode(prompt, mode='flat')
    else:
        ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)

    eos_id = (getattr(tokenizer, 'eos_token_id', None)
              or getattr(tokenizer, 'token2id', {}).get('</s>', None))

    out = model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_k          = 40,
        eos_token_id   = eos_id,
    )
    new_ids = out[0, len(ids):].tolist()
    if hasattr(tokenizer, 'decode') and hasattr(tokenizer, 'token2id'):
        return tokenizer.decode(new_ids)
    return tokenizer.decode(new_ids, skip_special_tokens=True)

print("Helper functions defined.")

In [ ]:
# ── Quick eval: perplexity + top-k + generation for every stage checkpoint ───
# Runs in ~2-3 min per stage in QUICK mode.

try:
    from semantic_eval import eval_perplexity, eval_topk_accuracy
except ImportError:
    raise ImportError(
        "Cannot import semantic_eval. "
        "Set REPO_ROOT in the config cell (needs the dataset/repo attached), "
        "or run: pip install -e git+https://github.com/sh20022002/small-Language-Model.git"
    )

scale = 0.2 if QUICK else 1.0
all_results = []   # list of dicts, one per checkpoint

checkpoints_to_eval = stage_ckpts.copy()
if final_ckpt.exists():
    checkpoints_to_eval.append(final_ckpt)

if not checkpoints_to_eval:
    print("No checkpoints to evaluate.")
else:
    for ckpt_path in checkpoints_to_eval:
        label = ckpt_path.stem.replace("_stage", "")  # e.g. 'tinystories_stories'
        print(f"\n{'='*60}")
        print(f"  Evaluating: {label}")
        print(f"{'='*60}")

        model = load_stage_model(ckpt_path)

        # Perplexity + BPC
        print("  Running perplexity...", end=" ", flush=True)
        ppl_res = eval_perplexity(model, tokenizer, device,
                                   n_examples=int(500 * scale))
        print(f"PPL={ppl_res.get('perplexity','?')}  BPC={ppl_res.get('bpc','?')}")

        # Top-k accuracy
        print("  Running top-k accuracy...", end=" ", flush=True)
        topk_res = eval_topk_accuracy(model, tokenizer, device,
                                       n_examples=int(300 * scale))
        print(f"Top-1={topk_res.get('top1_acc','?')}%  Top-5={topk_res.get('top5_acc','?')}%")

        # Generation sample
        prompt = pick_prompt(label)
        print(f"\n  Prompt: {prompt!r}")
        try:
            continuation = generate(model, prompt)
            print(f"  Output: {prompt}{continuation}")
        except Exception as e:
            print(f"  [generation error] {e}")
            continuation = ""

        all_results.append({
            "stage":      label,
            "perplexity": ppl_res.get("perplexity", float("nan")),
            "bpc":        ppl_res.get("bpc",        float("nan")),
            "top1_acc":   topk_res.get("top1_acc",  float("nan")),
            "top5_acc":   topk_res.get("top5_acc",  float("nan")),
            "prompt":     prompt,
            "generation": continuation,
        })

        del model   # free GPU memory before loading next checkpoint
        if device.type == "cuda":
            torch.cuda.empty_cache()

print("\nQuick eval complete.")

In [ ]:
# ── Comparison table across all stages ───────────────────────────────────────
if not all_results:
    print("No results to display.")
else:
    header = f"{'Stage':<35} {'PPL':>8} {'BPC':>6} {'Top-1%':>8} {'Top-5%':>8}"
    print("\n" + "=" * len(header))
    print("  STAGE COMPARISON")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for r in all_results:
        print(f"  {r['stage']:<33} {r['perplexity']:>8.1f} {r['bpc']:>6.3f} "
              f"{r['top1_acc']:>7.1f}% {r['top5_acc']:>7.1f}%")
    print("=" * len(header))
    print("  (lower PPL/BPC is better; higher top-k% is better)")
    print(f"  Reference: GPT-2-small  PPL=29.4  BPC=1.0  Top-1≈35%  Top-5≈60%")

    # Loss curve plots (if saved during training)
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg

    figs = sorted(Path(OUTPUT_DIR).glob("*_loss.png"))
    if figs:
        cols = min(3, len(figs))
        rows = (len(figs) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, fp in zip(axes.flat, figs):
            ax.imshow(mpimg.imread(str(fp)))
            ax.set_title(fp.stem.replace("_loss", ""), fontsize=10)
            ax.axis("off")
        plt.suptitle("Training loss curves per stage", fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("\nNo *_loss.png files found in OUTPUT_DIR.")

In [ ]:
# ── Full semantic eval: BLiMP + LAMBADA (optional, ~10 min/stage) ─────────────
# Skip this cell for a quick check. Run it after confirming basic metrics look good.

from semantic_eval import eval_blimp, eval_lambada, print_report

full_results = []

for ckpt_path in checkpoints_to_eval:
    label = ckpt_path.stem.replace("_stage", "")
    print(f"\n{'='*60}")
    print(f"  Full eval: {label}")
    print(f"{'='*60}")

    model = load_stage_model(ckpt_path)

    print("  BLiMP grammaticality...", end=" ", flush=True)
    blimp_res = eval_blimp(model, tokenizer, device,
                            n_per_phenomenon=int(100 * scale))
    print(f"overall={blimp_res.get('overall','?')}%")

    print("  LAMBADA last-word...", end=" ", flush=True)
    lambada_res = eval_lambada(model, tokenizer, device,
                                n_examples=int(500 * scale))
    print(f"top1={lambada_res.get('top1_acc','?')}%  top5={lambada_res.get('top5_acc','?')}%")

    full_results.append({
        "stage":         label,
        "blimp_overall": blimp_res.get("overall", float("nan")),
        "lambada_top1":  lambada_res.get("top1_acc", float("nan")),
        "lambada_top5":  lambada_res.get("top5_acc", float("nan")),
    })

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

# Summary
if full_results:
    header2 = f"{'Stage':<35} {'BLiMP%':>8} {'LAMBADA-1%':>11} {'LAMBADA-5%':>11}"
    print("\n" + "=" * len(header2))
    print("  FULL EVAL COMPARISON")
    print("=" * len(header2))
    print(header2)
    print("-" * len(header2))
    for r in full_results:
        print(f"  {r['stage']:<33} {r['blimp_overall']:>7.1f}% "
              f"{r['lambada_top1']:>10.1f}% {r['lambada_top5']:>10.1f}%")
    print("=" * len(header2))
    print("  Reference: GPT-2-small  BLiMP≈67%  LAMBADA-top1≈45%")

In [ ]:
# ── Interactive generation from any checkpoint ────────────────────────────────
# Edit CKPT_NAME and PROMPT, then re-run this cell.

CKPT_NAME   = "final_model.pt"       # filename in OUTPUT_DIR, e.g. 'tinystories_stories_stage.pt'
PROMPT      = "Once upon a time"
MAX_TOKENS  = 150
TEMPERATURE = 0.8
N_SAMPLES   = 3

ckpt = Path(OUTPUT_DIR) / CKPT_NAME
if not ckpt.exists():
    print(f"Not found: {ckpt}")
    print("Available:", [p.name for p in Path(OUTPUT_DIR).glob("*.pt")])
else:
    model = load_stage_model(ckpt)
    print(f"Checkpoint: {CKPT_NAME}")
    print(f"Prompt:     {PROMPT!r}")
    print()
    for i in range(N_SAMPLES):
        out = generate(model, PROMPT, max_new_tokens=MAX_TOKENS, temperature=TEMPERATURE)
        print(f"--- Sample {i+1} ---")
        print(PROMPT + out)
        print()